## 0. Kernel setup (run in a terminal, not in this notebook)

Before launching this notebook, create and select a conda environment kernel (`2ndWorkshop`).

### Purdue Gilbreth cluster

```bash
module load conda
conda-env-mod create -n ENV_NAME_HERE -j
module use $HOME/privatemodules
module load conda-env/ENV_NAME_HERE
```

Replace `ENV_NAME_HERE` with your environment name (`2ndWorkshop`), then select the matching kernel in Jupyter before running the cells below.

Check the env and kernel were created:
```bash
conda env list
jupyter kernelspec list
```

### Local machine (conda)

```bash
conda create -n 2ndWorkshop python=3.10 pip -y
conda activate 2ndWorkshop
pip install ipykernel
python -m ipykernel install --user --name 2ndWorkshop --display-name "Python (2ndWorkshop)"
```

Select the `Python (2ndWorkshop)` kernel, run the install cell below once, then restart the kernel.

**Note:** This notebook (Part 1) only needs local Hugging Face models — no API keys or
external servers required. **Part 2** (`Workshop2_Part2_Agent.ipynb`) is where you'll
need an LLM backend that supports tool calling (Ollama, OpenAI, or Purdue GenAI).


# Workshop 2, Part 1: Retrieval-Augmented Generation (RAG)

This is **Part 1 of 2** for Workshop 2. Build a RAG pipeline over a small Purdue course
knowledge base: embed documents, index them with FAISS, retrieve the most relevant
passages for a query, and generate a grounded answer with a local LLM.

**What you'll do:**
1. Load and inspect the course knowledge base
2. Embed documents and build a FAISS index
3. Retrieve the top-k relevant passages for a query
4. Build a chat-templated RAG prompt (same trick as Workshop 1)
5. Load a small local instruct model and answer real questions with cited sources

**Part 2** (`Workshop2_Part2_Agent.ipynb`) reuses this same retrieval approach, but wraps
it in tools an autonomous LangGraph agent can call: the Boilermaker TA.


## 0. Install dependencies

Run once, then restart the kernel.

**Use `%pip install`, not `!pip install`.** `!pip` runs whatever `pip` happens to be
first on your shell's `PATH`, which on clusters (and sometimes locally) is not
necessarily the kernel's own environment -- so the install can silently land in the
wrong place and the notebook still fails with `ModuleNotFoundError` afterward. `%pip` is
a magic command that always installs into the environment backing the *running kernel*
(it's equivalent to `sys.executable -m pip`), so what you install is guaranteed to be
what the notebook can import.

In [2]:
%pip install torch transformers accelerate sentence-transformers faiss-cpu langchain-text-splitters


Note: you may need to restart the kernel to use updated packages.


---
# Part A: Retrieval-Augmented Generation (RAG)

RAG = embed documents → store in a vector index → at query time, retrieve the most relevant chunks → pass them as context to the LLM.

## A1. Imports and data loading

In [3]:
import os
import json
from pathlib import Path
from typing import List

import faiss
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter

BASE_DIR = Path(".").resolve()
DATA_DIR = BASE_DIR / "boilermaker_ta_data"

print("Data directory:", DATA_DIR)
print("Files:", list(DATA_DIR.iterdir()) if DATA_DIR.exists() else "NOT FOUND")

/Users/elhambarezi/miniconda3/envs/2ndWorkshop/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data directory: /Users/elhambarezi/Desktop/next workshop/2ndWorkshop/boilermaker_ta_data
Files: [PosixPath('/Users/elhambarezi/Desktop/next workshop/2ndWorkshop/boilermaker_ta_data/purdue_calendar.json'), PosixPath('/Users/elhambarezi/Desktop/next workshop/2ndWorkshop/boilermaker_ta_data/knowledge_base.json')]


## A2. Load the knowledge base corpus

In [4]:
def _load_corpus():
    path = DATA_DIR / "knowledge_base.json"
    if not path.exists():
        # Fallback so the notebook runs even without the data file
        return [
            {
                "title": "Fallback: RAG Intro",
                "text": "This is a fallback document. Add boilermaker_ta_data/knowledge_base.json for workshop data.",
            }
        ]
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


CORPUS = _load_corpus()
print(f"Loaded {len(CORPUS)} documents")
print("First doc:", CORPUS[0]["title"])

Loaded 3 documents
First doc: CS 182 Syllabus


## A3. Build the FAISS retriever

Steps:
1. **Chunk** each document
2. **Embed** each chunk with `all-MiniLM-L6-v2` (384-dim vectors)
3. **L2-normalize** vectors so inner product = cosine similarity
4. **Index** in a FAISS `IndexFlatIP` (brute-force inner product search)

In production, save/load the index from disk to avoid re-embedding on every startup.

**Chunking runs unconditionally, for short and long documents alike.** The chunker uses
LangChain's `RecursiveCharacterTextSplitter` (`chunk_size=1200` / `chunk_overlap=300`
characters) instead of a naive fixed-size word window. It tries a cascade of separators: paragraph breaks, then sentences, then words; and only falls back to a hard
character cut when a piece still doesn't fit, so chunks tend to break at natural
boundaries instead of mid-sentence. For text shorter than `chunk_size`, it's a no-op:
the splitter just returns the whole document as a single chunk, so there's no separate
"skip chunking" path to maintain: one code path handles our short paragraph-length
course docs and long real-world files (PDFs, wikis, full lecture notes) the same way.

Each chunk also gets a small `Title: ...\nChunk: N\n` header prepended before embedding,
so it carries its source and position even when retrieved on its own, away from the rest
of the document. 


In [5]:
def chunk_text(text: str, chunk_size: int = 1200, chunk_overlap: int = 300) -> List[str]:
    """Split text into chunks using LangChain's RecursiveCharacterTextSplitter.

    Unlike a fixed word-count window, this splitter tries a cascade of separators
    (paragraph breaks, then sentences, then words) and only falls back to a hard
    character cut when a chunk still doesn't fit, so chunks tend to break at natural
    boundaries instead of mid-sentence. chunk_size / chunk_overlap are in characters.
    Short text (<= chunk_size chars) comes back as a single chunk, so this is safe to
    call on every document regardless of length.
    """
    # chunk_size is the maximum number of characters in a chunk, and chunk_overlap is the number of characters that overlap between chunks. 
    # Small chunks maximize retrieval precision but expand index size, while large chunks provide richer context to the LLMs but increases the chance of retrieving irrelevant text.
    # Small chunk_overlap minimizes redundant data and token costs but risks splitting and losing boundary context, whereas big overlap guarantees seamless context preservation at the cost of duplicate text and wasted LLM prompt space.
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return splitter.split_text(text)


def build_retriever(model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
    embedder = SentenceTransformer(model_name)

    texts, metadatas = [], []
    for i, doc in enumerate(CORPUS):
        title = doc.get("title") or f"doc{i}"
        for chunk_num, chunk in enumerate(chunk_text(doc["text"]), start=1):
            # Prepend a small header so each chunk carries its source title and
            # position even when retrieved on its own, away from the rest of the doc. 
            # This is very helpful for multiple-docs RAG and source citing.
            header = f"Title: {title}\nChunk: {chunk_num}\n"
            texts.append(header + chunk)
            metadatas.append({"title": title, "chunk": chunk_num})

    embeddings = embedder.encode(texts, convert_to_numpy=True, show_progress_bar=True)  # converting human language text into numerical vectors (embeddings) that capture semantic meaning, so that similar texts have similar embeddings. 
    if embeddings.ndim == 1:
        embeddings = embeddings.reshape(1, -1)

    faiss.normalize_L2(embeddings)          # normalize so dot product = cosine sim
    index = faiss.IndexFlatIP(embeddings.shape[1]) # create FAISS index for inner product (cosine sim) 
    index.add(embeddings) 

    return {"embedder": embedder, "index": index, "texts": texts, "metadatas": metadatas}


retriever = build_retriever()
print(f"FAISS index has {retriever['index'].ntotal} vectors of dim {retriever['index'].d}")

Batches: 100%|██████████| 1/1 [00:00<00:00,  8.71it/s]

FAISS index has 3 vectors of dim 384


## A4. Retrieve documents for a query

In [6]:
def retrieve_docs(retriever, query: str, k: int = 3): # Retrieve top-k most similar documents to a query
    q_emb = retriever["embedder"].encode(query, convert_to_numpy=True)  # use the same embedder to encode the query and docs.
    if q_emb.ndim == 1:
        q_emb = q_emb.reshape(1, -1)
    faiss.normalize_L2(q_emb)
    scores, ids = retriever["index"].search(q_emb, min(k, len(retriever["texts"])))
    return [
        {
            "text":     retriever["texts"][idx],
            "metadata": retriever["metadatas"][idx],
            "score":    float(scores[0][i]),
        }
        for i, idx in enumerate(ids[0])
    ]


# Test retrieval
query = "When are the office hours for this course?"
docs  = retrieve_docs(retriever, query, k=3)
for d in docs:
    print(f"[{d['score']:.3f}] {d['metadata']['title']}")
    print("  ", d["text"][:120], "...\n")

query = "what is the course platform?"
docs  = retrieve_docs(retriever, query, k=3)
for d in docs:
    print(f"[{d['score']:.3f}] {d['metadata']['title']}")
    print("  ", d["text"][:120], "...\n")

[0.428] Course Resources
   Title: Course Resources
Chunk: 1
The course uses GitHub Classroom, Piazza for questions, and a shared lecture notes repo ...

[0.419] Exam Policies
   Title: Exam Policies
Chunk: 1
Purdue exam weeks are in mid-October and mid-November. Makeup exams require prior approval ...

[0.379] CS 182 Syllabus
   Title: CS 182 Syllabus
Chunk: 1
CS 182 is a Purdue undergraduate course focused on software development. Assignments are ...

[0.461] Course Resources
   Title: Course Resources
Chunk: 1
The course uses GitHub Classroom, Piazza for questions, and a shared lecture notes repo ...

[0.292] CS 182 Syllabus
   Title: CS 182 Syllabus
Chunk: 1
CS 182 is a Purdue undergraduate course focused on software development. Assignments are ...

[0.173] Exam Policies
   Title: Exam Policies
Chunk: 1
Purdue exam weeks are in mid-October and mid-November. Makeup exams require prior approval ...



## A5. Build a RAG prompt and generate an answer

The retrieved context is inserted into the prompt so the LLM can ground its answer in
retrieved facts instead of guessing.

Just like Workshop 1's `build_messages` + `apply_chat_template`, we render a
`system` / `user` message pair through the model's native chat template, instruct
models answer far more reliably this way than from a raw concatenated string.

The preview below can't use the real chat template yet because the tokenizer is loaded
in **A6**, one section from now — it prints the plain-text fallback. **A7** calls
`make_prompt` again with the loaded tokenizer, so the actual queries do get the
chat-templated version.


In [7]:
def make_prompt(query: str, context: str, tokenizer=None) -> str:
    # Same trick as Workshop 1: instruct models follow instructions far more reliably
    # when the prompt is rendered through their native chat template instead of a
    # raw string glued together with "Context:" / "Question:" labels.
    messages = [
        {
            "role": "system",
            "content": "You are a course TA. Answer using only the provided context. "
            "If the answer isn't in the context, say you don't know.",
        },
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
    ]
    if tokenizer is not None:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    # Fallback for base (non-chat) models, or when no tokenizer is available yet
    return f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer in a concise, factual way."


# Preview without a tokenizer (fallback format) — A7 renders the real chat-templated version
context = "\n\n".join([d["text"] for d in docs])
prompt  = make_prompt(query, context)
print(prompt)


Context:
Title: Course Resources
Chunk: 1
The course uses GitHub Classroom, Piazza for questions, and a shared lecture notes repository. Instructors recommend starting project planning at least three weeks before each milestone.

Title: CS 182 Syllabus
Chunk: 1
CS 182 is a Purdue undergraduate course focused on software development. Assignments are due on Wednesdays at 11:59 PM. Office hours are Tuesdays 3-5 PM and Thursdays 2-4 PM. The final project presentation is scheduled in the last week of classes.

Title: Exam Policies
Chunk: 1
Purdue exam weeks are in mid-October and mid-November. Makeup exams require prior approval. Students are encouraged to schedule study groups at least one week before the exam.

Question: what is the course platform?

Answer in a concise, factual way.


## A6. Load the local LLM

Same default as Workshop 1 — **`Qwen/Qwen2.5-0.5B-Instruct`** — small enough to run on a
laptop CPU in a few seconds, so the RAG demo is reliable for everyone in the room.

`LOCAL_MODEL` env var lets you swap models without changing code.
`device_map="auto"` places layers on GPU if available, CPU otherwise.
`bfloat16` is only requested when CUDA is available — avoids issues on CPU-only machines.


In [8]:
def get_llm():
    # Swap example: LOCAL_MODEL="mistralai/Mistral-7B-Instruct-v0.3" (needs a GPU)
    local_model_name = os.environ.get("LOCAL_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
    tokenizer    = AutoTokenizer.from_pretrained(local_model_name)
    model_kwargs = {"device_map": "auto"}
    if torch.cuda.is_available():
        model_kwargs["torch_dtype"] = torch.bfloat16   # bfloat16 only safe with CUDA

    local_model = AutoModelForCausalLM.from_pretrained(local_model_name, **model_kwargs)
    # Some checkpoints ship a generation_config with a leftover max_length default,
    # which then conflicts with the max_new_tokens passed below. Clear it so
    # max_new_tokens is the only length control in play.
    local_model.generation_config.max_length = None
    return pipeline("text-generation", model=local_model, tokenizer=tokenizer, max_new_tokens=256)


print("Loading LLM... (a few seconds on CPU with the default model)")
llm = get_llm()
print("LLM ready.")


Loading LLM... (a few seconds on CPU with the default model)


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 44006.81it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM ready.


## A7. Run RAG queries end-to-end

In [9]:
def answer_queries(queries: List[str], llm, retriever):
    for query in queries:
        print(f"\n=== QUERY: {query}\n")
        docs    = retrieve_docs(retriever, query, k=1)
        context = "\n\n".join([d["text"] for d in docs])
        # llm.tokenizer: transformers pipelines expose the tokenizer they were built
        # with, so we can render the same chat template used during A6's load.
        prompt  = make_prompt(query, context, tokenizer=llm.tokenizer)
        output  = llm(prompt, return_full_text=False)[0]["generated_text"] 
        print("Answer:\n", output)
        print("Sources:\n", [d["metadata"] for d in docs])


queries = [
    "When are the office hours for this course?",
    "When are assignments due and what is the usual deadline?",
    "When are the exam weeks scheduled?",
    "What tools and resources does the course recommend for projects?",
    "When is the final project presentation scheduled?",
]

answer_queries(queries, llm, retriever)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== QUERY: When are the office hours for this course?



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer:
 Office hours for this course are by appointment only.
Sources:
 [{'title': 'Course Resources', 'chunk': 1}]

=== QUERY: When are assignments due and what is the usual deadline?



[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer:
 Assignments are typically due around October 30th or November 2nd. The usual deadline for submitting assignments is usually by December 7th.
Sources:
 [{'title': 'Exam Policies', 'chunk': 1}]

=== QUERY: When are the exam weeks scheduled?



[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer:
 The exam weeks are scheduled in mid-October and mid-November.
Sources:
 [{'title': 'Exam Policies', 'chunk': 1}]

=== QUERY: What tools and resources does the course recommend for projects?

Answer:
 The course recommends starting project planning at least three weeks before each milestone.
Sources:
 [{'title': 'Course Resources', 'chunk': 1}]

=== QUERY: When is the final project presentation scheduled?



[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer:
 The final project presentation is scheduled to be held on June 10th.
Sources:
 [{'title': 'Course Resources', 'chunk': 1}]


---
## Next: Part 2 — Agentic AI

Open **`Workshop2_Part2_Agent.ipynb`** to continue. It reuses the same embeddings + FAISS
approach from this notebook, but wraps it in tools that an autonomous LangGraph agent
decides when to call — plus a calendar tool and a notification-writing tool. That's the
Boilermaker TA.
